# Pittsburgh Dual-Model Proximity Analysis

**Project:** Odor-Complaint Weather Risk Prediction
**Purpose:** Engineer emission-source proximity features (distance, exponential decay,
continuous wind-alignment) and compare a **weather-only** logit model (Model A) against a
**proximity-enhanced** logit model (Model B).

**Key output:** `Pittsburgh Data/model_coeffs_pittsburgh.json` — coefficients ported into the live Calvert City forecaster.

---


## Section 1 — Setup & Data Loading

Load the merged Pittsburgh hourly CSV, apply column mapping, derive zip centroids,
aggregate to a daily zip-level panel, and engineer base features.


In [4]:
import json
import math
import warnings

import matplotlib
matplotlib.use("Agg")   # comment out if running interactively in Jupyter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

# Optional packages
try:
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_curve, roc_auc_score
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
    print("sklearn not found – CV ROC-AUC will be skipped")

try:
    import holidays
    HAS_HOLIDAYS = True
except ImportError:
    HAS_HOLIDAYS = False
    print("holidays package not found – is_holiday will be 0")


In [5]:
DATA_PATH   = "open-meteo-smell-merged.csv"
PLOT_PREFIX = "Dual_Model_Proximity_Analysis"

COL_MAP = {
    "time":                          "datetime",
    "smell_report_count":            "complaints",
    "temperature_2m (°F)":           "temperature",
    "relative_humidity_2m (%)":      "relative_humidity",
    "wind_speed_10m (mp/h)":         "wind_speed",
    "wind_direction_10m (°)":        "wind_direction",
    "rain (inch)":                   "precipitation",
    "dew_point_2m (°F)":             "dew_point",
    "vapour_pressure_deficit (kPa)": "vapor_pressure",
    "surface_pressure (hPa)":        "atmospheric_pressure",
    "shortwave_radiation (W/m²)":    "solar_radiation",
    "sunshine_duration (s)":         "sunshine_duration",
    "boundary_layer_height (ft)":    "boundary_layer_height",
}

print(f"Loading {DATA_PATH} …")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"  Raw shape: {df_raw.shape}")

rename_actual = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
df_raw = df_raw.rename(columns=rename_actual)

numeric_cols = ["complaints", "temperature", "relative_humidity", "wind_speed",
                "wind_direction", "precipitation", "atmospheric_pressure",
                "solar_radiation", "boundary_layer_height", "smell_value_average"]
for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], errors="coerce")
df_raw["date"]     = df_raw["datetime"].dt.date
df_raw["zipcode"]  = df_raw["zipcode"].astype(str)
print(f"  Zipcodes: {df_raw['zipcode'].nunique()}")
print(f"  Date range: {df_raw['date'].min()} to {df_raw['date'].max()}")


Loading open-meteo-smell-merged.csv …
  Raw shape: (2472561, 26)
  Zipcodes: 34
  Date range: 2017-12-31 to 2026-06-06


In [6]:
# Zip centroids
zip_centroids = (
    df_raw.groupby("zipcode")[["latitude", "longitude"]]
    .mean().reset_index()
    .rename(columns={"latitude": "lat", "longitude": "lon"})
)
print(f"Zip centroids computed for {len(zip_centroids)} zips")
zip_centroids.head()


Zip centroids computed for 34 zips


,zipcode,lat,lon
0,15025,40.295604,-79.910408
1,15034,40.350453,-79.889960
2,15037,40.290127,-79.868452
3,15045,40.325515,-79.886523
4,15102,40.308982,-80.048858


In [7]:
def circular_wind_mean(series):
    """Circular mean for wind direction (degrees)."""
    rads = np.radians(series.dropna())
    if len(rads) == 0:
        return np.nan
    u = np.mean(np.sin(rads))
    v = np.mean(np.cos(rads))
    return (np.degrees(np.arctan2(u, v)) + 360) % 360

# Drop true no-report padding rows (complaints==0 AND smell_value_average is NaN)
mask = (df_raw["complaints"] == 0) & (df_raw["smell_value_average"].isna())
df_filtered = df_raw[~mask].copy()
print(f"Dropped {mask.sum():,} padding rows; {len(df_filtered):,} remain")


Dropped 2,399,504 padding rows; 73,057 remain


In [8]:
agg_spec = {
    "complaints":            "sum",
    "temperature":           ["mean", "min", "max"],
    "precipitation":         "sum",
    "wind_speed":            "mean",
    "atmospheric_pressure":  "mean",
    "solar_radiation":       "mean",
    "boundary_layer_height": "mean",
    "relative_humidity":     "mean",
    "smell_value_average":   "mean",
}
agg_spec = {k: v for k, v in agg_spec.items() if k in df_filtered.columns}

grouped   = df_filtered.groupby(["zipcode", "date"])
daily_zip = grouped.agg(agg_spec)
daily_zip.columns = ["_".join(c).strip("_") if isinstance(c, tuple) else c
                     for c in daily_zip.columns]
daily_zip = daily_zip.reset_index()

# Circular wind direction
wind_circ = grouped["wind_direction"].apply(circular_wind_mean).reset_index()
wind_circ.columns = ["zipcode", "date", "wind_direction"]
daily_zip = daily_zip.merge(wind_circ, on=["zipcode", "date"], how="left")
daily_zip.columns = [c.replace(" ", "_") for c in daily_zip.columns]

flat_rename = {
    "temperature_mean": "temperature",  "temperature_min": "temperature_min",
    "temperature_max": "temperature_max", "precipitation_sum": "precipitation",
    "wind_speed_mean": "wind_speed",    "atmospheric_pressure_mean": "atmospheric_pressure",
    "solar_radiation_mean": "solar_radiation",
    "boundary_layer_height_mean": "boundary_layer_height",
    "relative_humidity_mean": "relative_humidity",
    "smell_value_average_mean": "smell_value_average",
    "complaints_sum": "complaints",
}
daily_zip = daily_zip.rename(columns={k: v for k, v in flat_rename.items()
                                       if k in daily_zip.columns})
print(f"Daily zip panel shape: {daily_zip.shape}")
daily_zip.head()


Daily zip panel shape: (36596, 14)


,zipcode,date,complaints,temperature,temperature_min,temperature_max,precipitation,wind_speed,atmospheric_pressure,solar_radiation,boundary_layer_height,relative_humidity,smell_value_average,wind_direction
0,15025,2018-01-03,1,28.300000,28.3,28.3,0.000,6.8,988.300000,449.000000,8557.309985,19.000000,3.0,197.0
1,15025,2018-01-10,1,39.700000,39.7,39.7,0.000,4.9,995.500000,87.000000,2798.615575,71.000000,3.0,150.0
2,15025,2018-01-14,2,8.100000,4.5,11.7,0.000,4.1,1008.150000,183.000000,2045.142782,68.500000,3.0,42.5
3,15025,2018-01-22,3,48.466667,45.4,53.5,0.008,8.0,985.733333,8.666667,1363.428521,85.666667,4.0,166.0
4,15025,2018-01-24,1,31.000000,31.0,31.0,0.000,8.6,992.400000,128.000000,12647.595549,61.000000,3.0,301.0


In [9]:
daily_zip["date"] = pd.to_datetime(daily_zip["date"])
daily_zip["diurnal_temperature_range"] = daily_zip["temperature_max"] - daily_zip["temperature_min"]
daily_zip["temperature_squared"]       = daily_zip["temperature"] ** 2

daily_zip["smell_value_average"]  = daily_zip["smell_value_average"].fillna(0)
daily_zip["weighted_odor_burden"] = daily_zip["complaints"] * daily_zip["smell_value_average"]
wob_mean = daily_zip["weighted_odor_burden"].mean()
daily_zip["is_odor_event"] = (daily_zip["weighted_odor_burden"] > wob_mean).astype(int)
print(f"ORI event rate: {daily_zip['is_odor_event'].mean():.3f}  (WOB threshold={wob_mean:.2f})")

# Day-of-week dummies (Monday = reference)
dow_map = {1: "tue", 2: "wed", 3: "thu", 4: "fri", 5: "sat", 6: "sun"}
for num, name in dow_map.items():
    daily_zip[f"dow_{name}"] = (daily_zip["date"].dt.dayofweek == num).astype(int)

if HAS_HOLIDAYS:
    us_hols = holidays.US()
    daily_zip["is_holiday"] = daily_zip["date"].apply(lambda d: int(d in us_hols))
else:
    daily_zip["is_holiday"] = 0

print(f"Final daily_zip shape: {daily_zip.shape}")


ORI event rate: 0.238  (WOB threshold=10.04)
Final daily_zip shape: (36596, 25)


## Section 2 — Proximity Feature Engineering

For each emission source, compute:
- **Haversine distance** (miles) from source to zip centroid
- **Exponential decay** `exp(−0.02 × dist)` — exposure proxy
- **Continuous wind alignment** (0–1) — 1 when wind blows directly from source to receptor

Then aggregate across sources to get `multi_source_exposure` and `wind_align_weighted`.


In [10]:
EMISSION_SOURCES = {
    "Clairton_Coke_Works": (40.2974, -79.8809),
    "Edgar_Thomson_Works":  (40.3922, -79.8550),
    "Irvin_Works":          (40.3644, -79.8944),
}

def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = (math.sin(dphi/2)**2
         + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2)
    return R * 2 * math.asin(math.sqrt(a))

def bearing(src_lat, src_lon, dst_lat, dst_lon):
    dy = dst_lat - src_lat
    dx = (dst_lon - src_lon) * math.cos(math.radians(src_lat))
    return (math.degrees(math.atan2(dx, dy)) + 360) % 360

def continuous_wind_alignment(wind_from_deg, bearing_deg):
    """
    Returns a value in [0, 1].
    1 = wind blowing directly from source toward receptor.
    """
    wind_toward = (wind_from_deg + 180) % 360
    angle_diff  = math.radians(wind_toward - bearing_deg)
    return (1 + math.cos(angle_diff)) / 2


In [11]:
# Merge zip centroids
daily_zip = daily_zip.merge(zip_centroids[["zipcode", "lat", "lon"]],
                             on="zipcode", how="left")

for src_name, (src_lat, src_lon) in EMISSION_SOURCES.items():
    print(f"  Computing features for {src_name} …")
    daily_zip[f"dist_{src_name}"] = daily_zip.apply(
        lambda r: haversine_miles(src_lat, src_lon, r["lat"], r["lon"]), axis=1)
    daily_zip[f"exp02_{src_name}"] = np.exp(-0.02 * daily_zip[f"dist_{src_name}"])
    daily_zip[f"bearing_{src_name}"] = daily_zip.apply(
        lambda r: bearing(src_lat, src_lon, r["lat"], r["lon"]), axis=1)
    daily_zip[f"wind_align_{src_name}"] = daily_zip.apply(
        lambda r: continuous_wind_alignment(
            r["wind_direction"], r[f"bearing_{src_name}"]), axis=1)

exp02_cols      = [f"exp02_{s}"      for s in EMISSION_SOURCES]
wind_align_cols = [f"wind_align_{s}" for s in EMISSION_SOURCES]
dist_cols       = [f"dist_{s}"       for s in EMISSION_SOURCES]

daily_zip["multi_source_exposure"] = daily_zip[exp02_cols].sum(axis=1)
total_exp = daily_zip[exp02_cols].sum(axis=1).replace(0, np.nan)
weighted_num = sum(daily_zip[f"exp02_{s}"] * daily_zip[f"wind_align_{s}"]
                   for s in EMISSION_SOURCES)
daily_zip["wind_align_weighted"] = weighted_num / total_exp
daily_zip["dist_nearest"]        = daily_zip[dist_cols].min(axis=1)

print(f"multi_source_exposure  : {daily_zip['multi_source_exposure'].min():.3f} – {daily_zip['multi_source_exposure'].max():.3f}")
print(f"wind_align_weighted    : {daily_zip['wind_align_weighted'].min():.3f} – {daily_zip['wind_align_weighted'].max():.3f}")
print(f"dist_nearest           : {daily_zip['dist_nearest'].min():.1f} – {daily_zip['dist_nearest'].max():.1f} miles")


  Computing features for Clairton_Coke_Works …
  Computing features for Edgar_Thomson_Works …
  Computing features for Irvin_Works …
multi_source_exposure  : 2.029 – 2.854
wind_align_weighted    : 0.002 – 0.998
dist_nearest           : 0.2 – 18.1 miles


## Section 3 — Binned Distribution Plots

Bin each proximity feature into 8 quantile groups and plot mean daily complaints per bin
with standard-error bars.


In [12]:
features_to_plot = {
    "dist_nearest":          "Distance to Nearest Source (miles)",
    "multi_source_exposure": "Multi-Source Exposure (Σ exp(−0.02·dist))",
    "wind_align_weighted":   "Exposure-Weighted Wind Alignment (0–1)",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Pittsburgh: Mean Daily Complaints by Proximity Feature Bin",
             fontsize=13, fontweight="bold")

for ax, (feat, label) in zip(axes, features_to_plot.items()):
    try:
        bins = pd.qcut(daily_zip[feat], q=8, duplicates="drop")
    except Exception:
        bins = pd.cut(daily_zip[feat], bins=8)
    grp = daily_zip.groupby(bins)["complaints"].agg(["mean", "sem"]).dropna()
    x   = range(len(grp))
    ax.bar(x, grp["mean"], yerr=grp["sem"], capsize=4,
           color="steelblue", alpha=0.8, edgecolor="white")
    ax.set_xticks(list(x))
    ax.set_xticklabels([str(b) for b in grp.index], rotation=45, ha="right", fontsize=7)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Mean Daily Complaints", fontsize=9)
    ax.set_title(feat.replace("_", " ").title(), fontsize=10)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section3_bins.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved.")


Plot saved.


## Section 4 — Feature Importance Screening

Compare four model specifications (weather-only, +proximity, +alignment, +both)
using both Logit and Poisson GLMs.  Displays Pseudo-R², AIC, and p-values for the
two new proximity features.


In [13]:
weather_vars = [
    "temperature", "temperature_squared", "solar_radiation",
    "relative_humidity", "wind_speed", "precipitation",
    "diurnal_temperature_range", "boundary_layer_height", "atmospheric_pressure",
    "dow_tue", "dow_wed", "dow_thu", "dow_fri", "dow_sat", "dow_sun", "is_holiday",
]
weather_vars = [v for v in weather_vars if v in daily_zip.columns]

model_specs = {
    "Weather Only":     weather_vars,
    "+Proximity":       weather_vars + ["multi_source_exposure"],
    "+Wind Alignment":  weather_vars + ["wind_align_weighted"],
    "+Both":            weather_vars + ["multi_source_exposure", "wind_align_weighted"],
}

target_logit   = "is_odor_event"
target_poisson = "complaints"
results_table  = []

for spec_name, feat_list in model_specs.items():
    sub = daily_zip[feat_list + [target_logit, target_poisson]].dropna()
    X   = sm.add_constant(sub[feat_list])
    y_b = sub[target_logit]
    y_p = sub[target_poisson]

    try:
        lf   = sm.Logit(y_b, X).fit(disp=0, method="bfgs", maxiter=300)
        lr2  = lf.prsquared; aic = lf.aic
        p_e  = lf.pvalues.get("multi_source_exposure", np.nan)
        p_a  = lf.pvalues.get("wind_align_weighted",  np.nan)
    except:
        lr2 = aic = p_e = p_a = np.nan

    try:
        pf   = sm.GLM(y_p, X, family=sm.families.Poisson()).fit(disp=0)
        p_r2 = 1 - pf.deviance / pf.null_deviance; p_aic = pf.aic
    except:
        p_r2 = p_aic = np.nan

    results_table.append({
        "Model":             spec_name,
        "Logit_PseudoR2":   round(lr2,  4),
        "Logit_AIC":        round(aic,  1),
        "Poisson_PseudoR2": round(p_r2, 4),
        "Poisson_AIC":      round(p_aic,1),
        "p(exposure)":      f"{p_e:.4f}" if not np.isnan(p_e) else "—",
        "p(alignment)":     f"{p_a:.4f}" if not np.isnan(p_a) else "—",
    })

pd.DataFrame(results_table)


,Model,Logit_PseudoR2,Logit_AIC,Poisson_PseudoR2,Poisson_AIC,p(exposure),p(alignment)
0,Weather Only,0.3870,24633.2,0.5425,137363.9,—,—
1,+Proximity,0.3892,24547.9,0.5463,136991.0,0.0000,—
2,+Wind Alignment,0.4056,23889.8,0.5629,135362.3,—,0.0000
3,+Both,0.4093,23744.6,0.5686,134806.4,0.0000,0.0000


## Section 5 — Dual-Model Full Comparison

### A. Fit both logit models on the complete-case sample


In [14]:
prox_vars = weather_vars + ["multi_source_exposure", "wind_align_weighted"]
sub_full  = daily_zip[list(set(prox_vars + [target_logit, target_poisson]))].dropna()
print(f"Complete-case rows: {len(sub_full):,}")

X_wa  = sm.add_constant(sub_full[weather_vars])
X_pe  = sm.add_constant(sub_full[prox_vars])
y_bin = sub_full[target_logit]

logit_weather_only       = sm.Logit(y_bin, X_wa).fit(disp=0, method="bfgs", maxiter=300)
logit_proximity_enhanced = sm.Logit(y_bin, X_pe).fit(disp=0, method="bfgs", maxiter=300)
poisson_wa               = sm.GLM(sub_full[target_poisson], X_wa,
                                   family=sm.families.Poisson()).fit(disp=0)
poisson_pe               = sm.GLM(sub_full[target_poisson], X_pe,
                                   family=sm.families.Poisson()).fit(disp=0)

delta_r2  = logit_proximity_enhanced.prsquared - logit_weather_only.prsquared
delta_aic = logit_weather_only.aic - logit_proximity_enhanced.aic
print(f"ΔPseudo-R²  (B − A) : {delta_r2:+.4f}")
print(f"ΔAIC        (A − B) : {delta_aic:+.1f}  (positive = Model B better)")


Complete-case rows: 36,596
ΔPseudo-R²  (B − A) : +0.0222
ΔAIC        (A − B) : +888.6  (positive = Model B better)


### B. Coefficient bar chart — top 14 features by magnitude

In [15]:
def top_n_coefs(fit_result, n=12):
    coefs = fit_result.params.drop("const", errors="ignore")
    return coefs.reindex(coefs.abs().nlargest(n).index)

coefs_b   = top_n_coefs(logit_proximity_enhanced, n=12)
coefs_a   = top_n_coefs(logit_weather_only,        n=12)
all_feat  = list(dict.fromkeys(list(coefs_b.index) + list(coefs_a.index)))[:14]
coefs_a_v = [logit_weather_only.params.get(f, 0)       for f in all_feat]
coefs_b_v = [logit_proximity_enhanced.params.get(f, 0) for f in all_feat]

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = np.arange(len(all_feat)); bar_h = 0.35
ax.barh(y_pos + bar_h/2, coefs_b_v, bar_h, label="Model B (Proximity-Enhanced)",
        color="steelblue", alpha=0.85)
ax.barh(y_pos - bar_h/2, coefs_a_v, bar_h, label="Model A (Weather Only)",
        color="coral",    alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels([f.replace("_", " ") for f in all_feat], fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Logit Coefficient", fontsize=11)
ax.set_title("Logit Coefficients: Weather-Only vs Proximity-Enhanced (top 14)", fontsize=11)
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5B_coefs.png", dpi=120, bbox_inches="tight")
plt.show()


### C. Predicted ORI probability distributions

In [16]:
pred_a = logit_weather_only.predict(X_wa)
pred_b = logit_proximity_enhanced.predict(X_pe)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pred_a, bins=60, alpha=0.55, label="Model A – Weather Only",
        color="coral",     density=True, edgecolor="white")
ax.hist(pred_b, bins=60, alpha=0.55, label="Model B – Proximity-Enhanced",
        color="steelblue", density=True, edgecolor="white")
ax.set_xlabel("Predicted ORI Probability", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Pittsburgh: Predicted ORI Probability Distributions", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5C_distributions.png", dpi=120, bbox_inches="tight")
plt.show()


### D. ROC curves + optional 5-fold CV AUC

In [17]:
fpr_a, tpr_a, _ = roc_curve(y_bin, pred_a)
fpr_b, tpr_b, _ = roc_curve(y_bin, pred_b)
auc_a = roc_auc_score(y_bin, pred_a)
auc_b = roc_auc_score(y_bin, pred_b)

if HAS_SKLEARN:
    scaler = StandardScaler()
    try:
        cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        lr_clf = LogisticRegression(max_iter=500, solver="lbfgs")
        cv_auc_a = cross_val_score(lr_clf,
                                   scaler.fit_transform(sub_full[weather_vars].fillna(0)),
                                   y_bin.values, cv=cv, scoring="roc_auc").mean()
        cv_auc_b = cross_val_score(lr_clf,
                                   scaler.fit_transform(sub_full[prox_vars].fillna(0)),
                                   y_bin.values, cv=cv, scoring="roc_auc").mean()
        print(f"5-fold CV ROC-AUC  A: {cv_auc_a:.4f}   B: {cv_auc_b:.4f}")
    except Exception as e:
        print(f"CV AUC failed: {e}")
        cv_auc_a = cv_auc_b = None
else:
    cv_auc_a = cv_auc_b = None

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_a, tpr_a, color="coral",     lw=2, label=f"Model A (AUC={auc_a:.3f})")
ax.plot(fpr_b, tpr_b, color="steelblue", lw=2, label=f"Model B (AUC={auc_b:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate",  fontsize=11)
ax.set_title("Pittsburgh ORI Logit — ROC Curves", fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section5D_roc.png", dpi=120, bbox_inches="tight")
plt.show()


5-fold CV ROC-AUC  A: 0.9056   B: 0.9095


### E. Full Model B statsmodels summary

In [18]:
print(logit_proximity_enhanced.summary())
print(f"\nKey proximity coefficients:")
print(f"  multi_source_exposure: {logit_proximity_enhanced.params['multi_source_exposure']:.6f}  "
      f"p={logit_proximity_enhanced.pvalues['multi_source_exposure']:.4f}")
print(f"  wind_align_weighted  : {logit_proximity_enhanced.params['wind_align_weighted']:.6f}  "
      f"p={logit_proximity_enhanced.pvalues['wind_align_weighted']:.4f}")


                           Logit Regression Results                           
Dep. Variable:          is_odor_event   No. Observations:                36596
Model:                          Logit   Df Residuals:                    36577
Method:                           MLE   Df Model:                           18
Date:                Thu, 25 Jun 2026   Pseudo R-squ.:                  0.4093
Time:                        11:36:57   Log-Likelihood:                -11853.
converged:                       True   LL-Null:                       -20065.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                       -12.6666      2.419     -5.235      0.000     -17.409      -7.925
temperature                   0.0183      0.006      2.902      0.004       0.006     

## Section 6 — Export Coefficients to JSON

> **Critical:** These coefficients are ported into the live Calvert City forecaster.
> `dow_*` and `is_holiday` flags are set to 0 for Calvert City predictions (de-biasing).


In [20]:
coeffs_wa = logit_weather_only.params.to_dict()
coeffs_pe = logit_proximity_enhanced.params.to_dict()

output = {
    "city": "Pittsburgh",
    "note": (
        "Coefficients from logit model trained on Pittsburgh zip-day panel "
        "(de-biased: dow/holiday vars set to 0 for Calvert City predictions). "
        "Proximity-enhanced includes multi_source_exposure and wind_align_weighted."
    ),
    "weather_only": {k: float(v) for k, v in coeffs_wa.items()},
    "proximity_enhanced": {k: float(v) for k, v in coeffs_pe.items()},
    "model_metrics": {
        "weather_only_aic":             float(logit_weather_only.aic),
        "proximity_enhanced_aic":       float(logit_proximity_enhanced.aic),
        "delta_aic":                    float(logit_weather_only.aic - logit_proximity_enhanced.aic),
        "weather_only_pseudo_r2":       float(logit_weather_only.prsquared),
        "proximity_enhanced_pseudo_r2": float(logit_proximity_enhanced.prsquared),
        "weather_only_auc":             float(auc_a),
        "proximity_enhanced_auc":       float(auc_b),
    },
}
if cv_auc_a is not None:
    output["model_metrics"]["weather_only_cv_auc"]       = float(cv_auc_a)
    output["model_metrics"]["proximity_enhanced_cv_auc"] = float(cv_auc_b)

json_path = "model_coeffs_pittsburgh.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {json_path}")
print(f"Key proximity coefficients:")
print(f"  multi_source_exposure: {coeffs_pe.get('multi_source_exposure', 'N/A')}")
print(f"  wind_align_weighted  : {coeffs_pe.get('wind_align_weighted',   'N/A')}")
print(f"\nΔPseudo-R² (B−A): {output['model_metrics']['proximity_enhanced_pseudo_r2'] - output['model_metrics']['weather_only_pseudo_r2']:+.4f}")
print(f"ΔAIC        (A−B): {output['model_metrics']['delta_aic']:+.1f}")


Saved model_coeffs_pittsburgh.json
Key proximity coefficients:
  multi_source_exposure: 1.3318487869144982
  wind_align_weighted  : 1.6800007018699348

ΔPseudo-R² (B−A): +0.0222
ΔAIC        (A−B): +888.6


## Section 8 — Hourly Case-Crossover Odor Model

### Motivation

The daily model (Sections 1–6) aggregates all weather to zip-day resolution.
The ⏱️ **Hourly tab** in the forecaster needs a model that can shape *within-day* risk.
A naive hourly logit would learn "when people are awake" (diurnal reporting bias) and
"when the sun is up" (solar ≈ hour-of-day collinearity), not actual meteorology.

### Method: time-stratified case-crossover

We use **conditional logistic regression** with strata = (year, month, hour-of-day).
Within each stratum, odor-report city-hours (cases) are compared to non-report city-hours
(controls) at the same hour of the same calendar month.  By conditioning on the stratum,
both confounders are differenced out by construction.

**Features used:** temperature, temperature², BLH, wind speed, RH, pressure, precipitation.  
**Features omitted:** solar radiation (near-collinear with hour-of-day, absorbed by strata),
DTR (daily by construction).

### Inference / anchoring

The model has no intercept (conditional logit).  At inference the 24 hourly z-values are
re-centered so their mean equals logit(daily_ORI / 100), anchoring the within-day *shape*
to the calibrated daily ORI without changing its level.

In [ ]:
from statsmodels.discrete.conditional_models import ConditionalLogit as _CL

HOURLY_FEATS = [
    "temperature", "temperature_squared",
    "boundary_layer_height", "wind_speed",
    "relative_humidity", "atmospheric_pressure",
    "precipitation",
]

# ── 8.1  City-hour panel ──────────────────────────────────────────────────────
print("8.1  Building city-hour panel …")
_weather_h_cols = ["temperature", "relative_humidity", "wind_speed",
                   "atmospheric_pressure", "solar_radiation",
                   "boundary_layer_height", "precipitation"]
_weather_h_cols = [c for c in _weather_h_cols if c in df_raw.columns]

_df_h = df_raw.copy()
_df_h["_wob"] = _df_h["complaints"].fillna(0) * _df_h["smell_value_average"].fillna(0)
_agg_h = {"complaints": "sum", "_wob": "sum"}
for _c in _weather_h_cols:
    _agg_h[_c] = "mean"

_df_ch = _df_h.groupby("datetime").agg(_agg_h).reset_index()
_df_ch.rename(columns={"_wob": "weighted_burden_h"}, inplace=True)
_df_ch["datetime"]  = pd.to_datetime(_df_ch["datetime"])
_df_ch["hour"]      = _df_ch["datetime"].dt.hour
_df_ch["month"]     = _df_ch["datetime"].dt.month
_df_ch["year"]      = _df_ch["datetime"].dt.year

_burden_thresh = _df_ch["weighted_burden_h"].mean()
_df_ch["is_event_h"] = (_df_ch["weighted_burden_h"] > _burden_thresh).astype(int)
_n_total_h = len(_df_ch)
_n_event_h = int(_df_ch["is_event_h"].sum())
print(f"  City-hours: {_n_total_h:,} total | {_n_event_h:,} events ({_n_event_h/_n_total_h*100:.1f}%)")

# ── 8.2  Case-crossover strata ────────────────────────────────────────────────
print("8.2  Building strata (year × month × hour-of-day) …")
_df_ch["stratum"] = (_df_ch["year"].astype(str) + "_"
                     + _df_ch["month"].astype(str).str.zfill(2) + "_"
                     + _df_ch["hour"].astype(str).str.zfill(2))
_s_stats = _df_ch.groupby("stratum")["is_event_h"].agg(["sum", "count"])
_valid_s  = _s_stats[(_s_stats["sum"] > 0) & (_s_stats["sum"] < _s_stats["count"])].index
_df_cc    = _df_ch[_df_ch["stratum"].isin(_valid_s)].copy()
print(f"  Valid strata: {_df_cc['stratum'].nunique():,}  |  CC dataset: {len(_df_cc):,} rows")

# ── 8.3  Feature engineering ──────────────────────────────────────────────────
_df_cc["temperature_squared"] = _df_cc["temperature"] ** 2
_cc_keep = HOURLY_FEATS + ["is_event_h", "stratum", "hour", "month"]
_cc_keep = [c for c in _cc_keep if c in _df_cc.columns]
_df_cc = _df_cc[_cc_keep].dropna()
print(f"  After NaN drop: {len(_df_cc):,} rows, {_df_cc['stratum'].nunique():,} strata")

_stratum_codes = pd.Categorical(_df_cc["stratum"]).codes

# ── 8.4  Fit model ────────────────────────────────────────────────────────────
# Primary: ConditionalLogit. Fallback: Logit + hour-of-day & month FE dummies
# (statistically equivalent when strata are large enough to avoid incidental-parameter bias).
print("\n8.4  Fitting model …")
_cl_fit_ok  = False
_fit_method = "unknown"
_cl_coeffs = {}; _cl_pvals = {}; _cl_cis = {}

try:
    _cl_m = _CL(endog=_df_cc["is_event_h"], exog=_df_cc[HOURLY_FEATS], groups=_stratum_codes)
    _cl_r = _cl_m.fit(maxiter=300, method="bfgs", disp=False)
    _cl_coeffs = _cl_r.params.to_dict()
    try:
        _cl_pvals = _cl_r.pvalues.to_dict()
        _ci_df    = _cl_r.conf_int()
        _cl_cis   = {k: (float(_ci_df.loc[k, 0]), float(_ci_df.loc[k, 1])) for k in _cl_coeffs}
    except Exception:
        _cl_pvals = {}; _cl_cis = {}
    _cl_fit_ok  = True
    _fit_method = "ConditionalLogit (year × month × hour-of-day strata)"
    print(f"  ConditionalLogit converged  (CIs available: {bool(_cl_cis)})")
except Exception as _e:
    print(f"  ConditionalLogit failed ({_e}); trying fallback …")

if not _cl_fit_ok:
    try:
        _h_dummies  = pd.get_dummies(_df_cc["hour"],  prefix="h",  drop_first=True).astype(float)
        _mo_dummies = pd.get_dummies(_df_cc["month"], prefix="mo", drop_first=True).astype(float)
        _X_fe = pd.concat([_df_cc[HOURLY_FEATS].reset_index(drop=True),
                           _h_dummies.reset_index(drop=True),
                           _mo_dummies.reset_index(drop=True)], axis=1)
        _X_fe = sm.add_constant(_X_fe)
        _res_fe = sm.Logit(_df_cc["is_event_h"].reset_index(drop=True), _X_fe).fit(
            maxiter=500, method="bfgs", disp=False)
        _cl_coeffs = {f: float(_res_fe.params[f])  for f in HOURLY_FEATS if f in _res_fe.params.index}
        _cl_pvals  = {f: float(_res_fe.pvalues[f]) for f in HOURLY_FEATS if f in _res_fe.pvalues.index}
        _ci_df_fe  = _res_fe.conf_int()
        _cl_cis    = {f: (float(_ci_df_fe.loc[f, 0]), float(_ci_df_fe.loc[f, 1]))
                      for f in HOURLY_FEATS if f in _ci_df_fe.index}
        _cl_fit_ok  = True
        _fit_method = "Logit + hour-of-day & month FE dummies (fallback)"
        print(f"  Fallback Logit converged  (pseudo-R² = {_res_fe.prsquared:.4f})")
    except Exception as _e2:
        print(f"  Fallback also failed: {_e2}")

if _cl_fit_ok:
    print(f"\n  Method: {_fit_method}")
    print(f"\n  {'Feature':<35} {'Coef':>9}  {'p-val':>7}  95% CI")
    print(f"  {'-'*70}")
    for _f in HOURLY_FEATS:
        _co = _cl_coeffs.get(_f, float("nan"))
        _pv = _cl_pvals.get(_f, float("nan"))
        _lo, _hi = _cl_cis.get(_f, (float("nan"), float("nan")))
        _sig = "*" if (not np.isnan(_pv) and _pv < 0.05) else " "
        print(f"{_sig} {_f:<35} {_co:+9.5f}  {_pv:7.4f}  [{_lo:+.5f}, {_hi:+.5f}]")

# ── 8.5  Robustness: sklearn unconditional logit + hour dummies ───────────────
print("\n8.5  Robustness check (sklearn logit + hour-of-day dummies) …")
_rob_coeffs = {}
if HAS_SKLEARN:
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    _h_dummies_rob = pd.get_dummies(_df_cc["stratum"].str[-2:].astype(int),
                                     prefix="h", drop_first=True)
    _X_rob = pd.concat([_df_cc[HOURLY_FEATS].reset_index(drop=True),
                        _h_dummies_rob.reset_index(drop=True)], axis=1)
    _y_rob = _df_cc["is_event_h"].values
    _scaler = StandardScaler()
    _X_rob_sc = _scaler.fit_transform(_X_rob)
    _feat_n = list(_X_rob.columns)
    _lr_rob = LogisticRegression(max_iter=500, solver="lbfgs").fit(_X_rob_sc, _y_rob)
    _rob_coeffs = {_feat_n[i]: _lr_rob.coef_[0][i] for i in range(len(_feat_n)) if _feat_n[i] in HOURLY_FEATS}
    print(f"  {'Feature':<35} {'Std coef (rob)':>14}  {'Match?'}")
    print(f"  {'-'*60}")
    for _f in HOURLY_FEATS:
        _rc = _rob_coeffs.get(_f, float("nan"))
        _cc_c = _cl_coeffs.get(_f, float("nan"))
        _match = "✓" if (not np.isnan(_rc) and not np.isnan(_cc_c) and np.sign(_rc) == np.sign(_cc_c)) else "?"
        print(f"  {_match} {_f:<35} {_rc:>+14.4f}")
else:
    print("  sklearn not available — skipping robustness check")

In [ ]:
# ── 8.6  Plots ────────────────────────────────────────────────────────────────
import json as _json

if _cl_fit_ok:
    # 8.6A  Odds-ratio forest plot
    _or_vals = {f: np.exp(_cl_coeffs[f]) for f in HOURLY_FEATS if f in _cl_coeffs}
    _or_lo   = {f: np.exp(_cl_cis[f][0]) for f in HOURLY_FEATS if f in _cl_cis}
    _or_hi   = {f: np.exp(_cl_cis[f][1]) for f in HOURLY_FEATS if f in _cl_cis}
    _feats_pl = [f for f in HOURLY_FEATS if f in _or_vals]
    _y_pos = np.arange(len(_feats_pl))

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, f in enumerate(_feats_pl):
        _color = "#ef4444" if _or_vals[f] > 1 else "#22c55e"
        ax.errorbar(_or_vals[f], i,
                    xerr=[[_or_vals[f] - _or_lo.get(f, _or_vals[f])],
                          [_or_hi.get(f, _or_vals[f]) - _or_vals[f]]],
                    fmt="o", color=_color, capsize=4, markersize=6)
        ax.text(_or_hi.get(f, _or_vals[f]) + 0.02, i,
                f"{_or_vals[f]:.3f}", va="center", fontsize=8.5)
    ax.axvline(1.0, color="#475569", linewidth=1, linestyle="--")
    ax.set_yticks(_y_pos)
    ax.set_yticklabels([f.replace("_", " ") for f in _feats_pl], fontsize=9)
    ax.set_xlabel("Odds Ratio (per unit change)", fontsize=10)
    ax.set_title(f"Hourly Case-Crossover Model — Odds Ratios\n({_fit_method})", fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{PLOT_PREFIX}_section8A_hourly_forest.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {PLOT_PREFIX}_section8A_hourly_forest.png")

    # 8.6B  Hourly vs daily coefficient comparison
    _daily_coeffs = logit_proximity_enhanced.params.to_dict()
    _shared_feats = [f for f in HOURLY_FEATS
                     if f in _cl_coeffs and f in _daily_coeffs]
    if _shared_feats:
        _std_h = {f: float(_df_cc[f].std()) for f in _shared_feats if f in _df_cc.columns}
        _std_d = {f: float(sub_full[f].std()) for f in _shared_feats if f in sub_full.columns}
        _std_h_coefs = [_cl_coeffs[f] * _std_h.get(f, 1) for f in _shared_feats]
        _std_d_coefs = [_daily_coeffs[f] * _std_d.get(f, 1) for f in _shared_feats]
        _y2 = np.arange(len(_shared_feats))
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(_y2 + 0.2, _std_h_coefs, 0.35, label="Hourly case-crossover",
                color="steelblue", alpha=0.85)
        ax.barh(_y2 - 0.2, _std_d_coefs, 0.35, label="Daily proximity model",
                color="coral", alpha=0.85)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_yticks(_y2)
        ax.set_yticklabels([f.replace("_", " ") for f in _shared_feats], fontsize=9)
        ax.set_xlabel("Standardized coefficient (β × σ)", fontsize=10)
        ax.set_title("Hourly vs Daily Model: Standardized Coefficients", fontsize=10)
        ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{PLOT_PREFIX}_section8B_hourly_vs_daily_coefs.png", dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved {PLOT_PREFIX}_section8B_hourly_vs_daily_coefs.png")

# 8.6C  Within-day shape curves
if _cl_fit_ok:
    _df_ch2 = _df_ch.dropna(subset=[f for f in HOURLY_FEATS if f != "temperature_squared"]).copy()
    _df_ch2["temperature_squared"] = _df_ch2["temperature"] ** 2
    _z_h = sum(_cl_coeffs.get(f, 0) * _df_ch2[f] for f in HOURLY_FEATS if f in _df_ch2.columns)

    _day_z_mean = _df_ch2.assign(_z=_z_h).groupby(_df_ch2["datetime"].dt.date)["_z"].transform("mean")
    _z_shifted = _z_h.values - _day_z_mean.values

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ax = axes[0]
    _hourly_mean_z = _df_ch2.assign(_z=_z_shifted).groupby(_df_ch2["datetime"].dt.hour)["_z"].mean()
    ax.plot(_hourly_mean_z.index, _hourly_mean_z.values, "o-", color="steelblue", linewidth=2)
    ax.axhline(0, color="#94a3b8", linewidth=1, linestyle="--")
    ax.set_xlabel("Hour of day", fontsize=10)
    ax.set_ylabel("Mean relative log-odds (anchor=0)", fontsize=10)
    ax.set_title("Average within-day risk shape\n(anchored to daily mean)", fontsize=10)
    ax.set_xticks(range(0, 24, 3))
    ax.grid(alpha=0.3)

    ax = axes[1]
    _top3 = sorted([f for f in HOURLY_FEATS if f in _df_ch2.columns and f != "temperature_squared"],
                   key=lambda f: abs(_cl_coeffs.get(f, 0)), reverse=True)[:3]
    for _f in _top3:
        _contribution = _cl_coeffs.get(_f, 0) * _df_ch2[_f]
        _hourly_c = _contribution.groupby(_df_ch2["datetime"].dt.hour).mean()
        ax.plot(_hourly_c.index, _hourly_c.values, "o-", linewidth=1.5,
                label=_f.replace("_", " "))
    ax.set_xlabel("Hour of day", fontsize=10)
    ax.set_ylabel("Mean contribution to log-odds", fontsize=10)
    ax.set_title("Top-3 driver contributions by hour", fontsize=10)
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{PLOT_PREFIX}_section8C_hourly_shape.png", dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved {PLOT_PREFIX}_section8C_hourly_shape.png")

# 8.6D  Case vs control distributions for top drivers
_top_feats = sorted([f for f in HOURLY_FEATS if f in _df_cc.columns and f != "temperature_squared"],
                    key=lambda f: abs(_cl_coeffs.get(f, 0)), reverse=True)[:3]
fig, axes = plt.subplots(1, len(_top_feats), figsize=(4 * len(_top_feats), 4))
for ax, _f in zip(axes, _top_feats):
    _cases = _df_cc[_df_cc["is_event_h"] == 1][_f].dropna()
    _ctrls = _df_cc[_df_cc["is_event_h"] == 0][_f].dropna()
    ax.hist(_ctrls, bins=40, alpha=0.55, label="Controls", color="#94a3b8", density=True)
    ax.hist(_cases, bins=40, alpha=0.55, label="Cases",    color="#ef4444", density=True)
    ax.set_xlabel(_f.replace("_", " "), fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title(_f.replace("_", " "), fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Case vs Control Distributions — Top Hourly Drivers", fontsize=11)
plt.tight_layout()
plt.savefig(f"{PLOT_PREFIX}_section8D_hourly_distributions.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved {PLOT_PREFIX}_section8D_hourly_distributions.png")

# ── 8.7  Export coefficients ──────────────────────────────────────────────────
if _cl_fit_ok:
    _hourly_export = {
        "note": (
            f"Hourly odor-risk model — {_fit_method}. "
            "Hour-of-day and calendar-month fixed effects remove diurnal reporting "
            "behavior and the solar/hour-of-day collinearity, yielding genuine sub-daily "
            "meteorological coefficients. At inference the 24 hourly z-values are "
            "re-centered so their mean equals logit(daily_ORI/100), anchoring the "
            "within-day shape to the calibrated daily ORI without changing its level."
        ),
        "coefficients":           {k: float(v) for k, v in _cl_coeffs.items()},
        "p_values":               {k: float(v) for k, v in _cl_pvals.items()},
        "confidence_intervals_95":{
            k: [float(v[0]), float(v[1])] for k, v in _cl_cis.items()
        },
        "features_used": HOURLY_FEATS,
        "features_omitted": {
            "solar_radiation": "near-collinear with hour-of-day, absorbed by strata",
            "diurnal_temperature_range": "daily by construction, not meaningful at hourly resolution",
        },
        "metadata": {
            "n_city_hours":          int(_n_total_h),
            "n_events":              int(_n_event_h),
            "event_rate_pct":        round(_n_event_h / _n_total_h * 100, 2),
            "n_valid_strata":        int(_df_cc["stratum"].nunique()),
            "n_case_crossover_obs":  int(len(_df_cc)),
        },
    }
    _h_json_path = "Pittsburgh Data/model_coeffs_hourly.json"
    with open(_h_json_path, "w") as _fh:
        _json.dump(_hourly_export, _fh, indent=2)
    print(f"\nSaved {_h_json_path}")
    print(f"  n_city_hours={_n_total_h:,}  n_events={_n_event_h:,}  n_strata={_df_cc['stratum'].nunique():,}")
else:
    print("Skipped export (model did not converge).")